# 00. 框架全景图

Agent 框架从「一页 Python 脚本」到「企业级平台」，覆盖了整个抽象光谱。理解每个框架的设计哲学，比记住 API 更重要。

> **检查点**：能用一句话描述 4 种框架编程模型（Graph / Handoff / Role / Event-driven）。

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


In [ ]:
# 同一个 3-step workflow，4 种框架风格对比
from dataclasses import dataclass

def workflow_raw(scenario):
    """Raw: 无框架，纯函数+dict"""
    state = {"goal": scenario, "results": []}
    state["results"].append(f"Research: found facts about {scenario}")
    state["results"].append(f"Summary: key findings documented")
    state["results"].append("Review: verified, no errors")
    return f"Raw: {len(state['results'])} steps, {len(str(state))} chars of state"

@dataclass
class GraphState:
    goal: str; step: str = "research"; research: str = ""; summary: str = ""; review: str = ""

def workflow_graph(scenario):
    """Graph: LangGraph 风格 — nodes + edges + state"""
    state = GraphState(goal=scenario)
    nodes = {
        "research": lambda s: GraphState(s.goal,"summarize",f"Found facts about {s.goal}"),
        "summarize": lambda s: GraphState(s.goal,"review",s.research,f"Summary of {s.goal}"),
        "review": lambda s: GraphState(s.goal,"done",s.research,s.summary,"Verified ✅"),
    }
    while state.step != "done":
        state = nodes[state.step](state)
    return f"Graph: {state.step}, research={bool(state.research)}"

@dataclass
class SDKAgent:
    name: str; instructions: str
    def run(self, input_text: str) -> str:
        return f"[{self.name}] processed: {input_text}"

def workflow_sdk(scenario):
    """SDK: OpenAI Agents 风格 — Agent + Handoff"""
    r = SDKAgent("researcher","Find facts").run(scenario)
    w = SDKAgent("writer","Synthesize").run(r)
    rev = SDKAgent("reviewer","Verify").run(w)
    return f"SDK: 3 agents chained, final={rev[:40]}..."

@dataclass
class CrewTask:
    desc: str; agent: str; expected: str

def workflow_crew(scenario):
    """Crew: CrewAI 风格 — Role + Task + Crew"""
    tasks = [CrewTask(f"Research {scenario}","Researcher","Fact list"),
             CrewTask("Write report","Writer","Markdown"),
             CrewTask("Review","Reviewer","Approval")]
    return f"Crew: {len(tasks)} tasks with role assignments"

scenario = "market analysis"
print("Same workflow, different styles:\n")
for name, fn in [("Raw",workflow_raw),("Graph",workflow_graph),("SDK",workflow_sdk),("Crew",workflow_crew)]:
    print(f"  {name:6s}: {fn(scenario)}")

## 抽象层次与取舍

```
Low-level ←──────────────────────────────→ High-level
  Raw loop       LangGraph    AutoGen    CrewAI      Agno
  (完全控制)    (graph+state)  (teams)   (roles)   (一行部署)
  灵活 ↑                                                ↑ 快
  复杂 ↑                                                ↑ 受限
```

> **检查点**：能为你自己的项目需求在抽象光谱上定位合适的框架层次。

## 真实 API 实验

「不用框架」的真实代价：手写一个三步 LLM 链，体会框架帮你省掉的样板（消息管理、错误处理、状态传递）。


In [ ]:
# 手写三步链（框架出现前的日常）
topic = "agent 记忆系统"

# step 1: research
r = chat(client, f"列出 {topic} 的 3 个要点，每点一行。", max_tokens=2000)
print("1) research done")

# step 2: 手动把上一步输出拼进下一步输入 — 框架帮你自动做的就是这个
w = chat(client, f"把以下要点写成 50 字摘要：\n{r}", max_tokens=1500)
print("2) summarize done")

# step 3: review
v = chat(client, f"检查这段摘要是否有事实错误，只回答 OK 或问题所在：\n{w}", max_tokens=1000)
print("3) review done")
print("\n结果:", v)
print("\n体会：3 步就要手动传 2 次状态；30 步的 agent 呢？这就是框架存在的理由。")


## 练习

1. 用 Raw 风格实现一个带「重试+回退」机制的 workflow（Graph 和 SDK 风格的对比见后续 notebook）。
2. 阅读你当前项目中的 Agent 代码，判断它在抽象光谱上的位置。应该往上还是往下迁移？
3. 思考：为什么没有一个框架能同时做到「极简 API」和「极致控制」？